# Customer Support Ticket Classifier

This notebook demonstrates how to build a customer support ticket classifier . The goal is to categorize incoming customer support tickets into appropriate categories for efficient handling and resolution.

## Objective
- Classify customer support tickets based on their content (e.g., Billing, Technical Issue, Account Management, General Inquiry, etc.)


## Steps
1. **Data Preparation:** Load and preprocess a dataset of customer support tickets.
2. **Exploratory Data Analysis:** Understand the distribution and nature of ticket categories.
3. **Feature Engineering:** Convert ticket text into features suitable for model training.
4. **Model Training:** Train classification models to predict ticket categories.
5. **Evaluation:** Assess model performance and analyze results.
6. **Deployment Example:** Show how to use the trained model to classify new support tickets.

---
*The following cells implement this workflow for customer support ticket classification.*

## Setup

In [ ]:
# Install dependencies
%pip install -q openai transformers peft bitsandbytes accelerate datasets huggingface_hub gradio scikit-learn matplotlib seaborn

zsh:1: command not found: pip


In [ ]:
import os
import json
import random
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=True)

print('Setup complete')

In [ ]:
# Configuration — change HF_USERNAME to your HuggingFace username
HF_USERNAME = "your-hf-username"    # <-- change this
DATASET_NAME = f"{HF_USERNAME}/support-ticket-dataset"
MODEL_NAME = f"{HF_USERNAME}/support-ticket-llama"
BASE_MODEL = "meta-llama/Llama-3.2-3B"

TICKET_CATEGORIES = ["Billing", "Technical Issue", "Account Management", "General Inquiry"]

print(f"Dataset: {DATASET_NAME}")
print(f"Fine-tuned model: {MODEL_NAME}")

---

# Part 1 — Generate Synthetic Training Data

We use GPT-4.1-mini to generate 2000 labelled customer support tickets.

The generated data is pushed to HuggingFace Hub and never committed to git.

In [ ]:
from openai import OpenAI
import time

client = OpenAI()

SYSTEM_PROMPT = """You are a senior customer support agent. Generate realistic customer support tickets.
Each ticket must be a brief description as a customer would submit (1-2 sentences).Respond ONLY with a JSON object containing a single key \"tickets\", whose value is a JSON array. Each item: {\"description\": \"...\", \"category\": \"...\"}.\nCategory must be exactly one of: Billing, Technical Issue, Account Management, General Inquiry."""

def generate_tickets(category: str, count: int, retries: int = 3) -> list[dict]:
    """Generate `count` synthetic support tickets for a given category."""
    prompt = f"""Generate {count} realistic customer support tickets that should be categorized as '{category}'.\nInclude a variety of: customer types, issues, urgency, and context.\nMake them realistic and diverse."""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4.1-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},
                max_tokens=4096,
                temperature=0.9
            )
            result = json.loads(response.choices[0].message.content)

            tickets = result.get("tickets", [])
            if not tickets:
                for v in result.values():
                    if isinstance(v, list):
                        tickets = v
                        break

            for t in tickets:
                t['category'] = category
            return tickets

        except (json.JSONDecodeError, KeyError, StopIteration) as e:
            print(f"  Attempt {attempt+1} failed: {e}. Retrying...")
            time.sleep(1)

    print(f"  All {retries} attempts failed for category={category}, count={count}. Returning empty.")
    return []

In [ ]:
# Generate 500 tickets per category = 2000 total
# Batches of 20 to avoid JSON truncation from hitting output token limits
all_tickets = []

for category in TICKET_CATEGORIES:
    print(f"Generating {category} tickets...")
    category_tickets = []
    for batch in range(25):  # 25 batches × 20 = 500 per category
        batch_tickets = generate_tickets(category, 20)
        category_tickets.extend(batch_tickets)
        print(f"  Batch {batch+1}/25 — {len(category_tickets)} {category} tickets so far")
    all_tickets.extend(category_tickets)

random.shuffle(all_tickets)
print(f"\nTotal tickets generated: {len(all_tickets)}")
print("Distribution:", {cat: sum(1 for t in all_tickets if t['category'] == cat) for cat in TICKET_CATEGORIES})

In [ ]:
# Preview a few examples
for category in TICKET_CATEGORIES:
    example = next(t for t in all_tickets if t['category'] == category)
    print(f"[{category}] {example['description']}")
    print()

In [ ]:
from datasets import Dataset, DatasetDict

# Split 80/10/10 train/val/test
n = len(all_tickets)
train_tickets = all_tickets[:int(n * 0.8)]
val_tickets   = all_tickets[int(n * 0.8):int(n * 0.9)]
test_tickets  = all_tickets[int(n * 0.9):]

dataset = DatasetDict({
    "train": Dataset.from_list(train_tickets),
    "validation": Dataset.from_list(val_tickets),
    "test": Dataset.from_list(test_tickets),
})

print(dataset)

# Push to HuggingFace Hub — not stored in git
dataset.push_to_hub(DATASET_NAME, private=True)
print(f"Dataset pushed to: https://huggingface.co/datasets/{DATASET_NAME}")

---

# Part 2 — Build Prompts and Inspect Base Model

Format each case as a user/assistant conversation for causal LM fine-tuning.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset(DATASET_NAME)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

In [ ]:
SYSTEM_MSG = """You are a customer support assistant. Classify the category of the support ticket.
Respond with exactly one word: Billing, Technical Issue, Account Management, or General Inquiry."""

def make_prompt(description: str, category: str = None) -> str:
    """Build a chat-template formatted prompt. Include answer if training."""
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",   "content": f"Classify this support ticket:\n{description}"}
    ]
    if category:  # training — include the answer
        messages.append({"role": "assistant", "content": category})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=(category is None))

# Preview
sample = dataset['train'][0]
print("=== TRAINING PROMPT ===")
print(make_prompt(sample['description'], sample['category']))
print("\n=== TEST PROMPT (no answer) ===")
print(make_prompt(dataset['test'][0]['description']))

In [ ]:
import matplotlib.pyplot as plt

# Token count distribution
token_counts = [
    len(tokenizer.encode(make_prompt(item['description'], item['category'])))
    for item in dataset['train']
]

print(f"Avg tokens: {sum(token_counts)/len(token_counts):.1f}")
print(f"Max tokens: {max(token_counts)}")
print(f"Min tokens: {min(token_counts)}")

plt.figure(figsize=(10, 4))
plt.title("Token count per training prompt")
plt.xlabel("Tokens")
plt.ylabel("Count")
plt.hist(token_counts, bins=30, color="steelblue", rwidth=0.8)
plt.tight_layout()
plt.show()

MAX_LENGTH = 256  # well above the max we see

In [ ]:
# Tokenize the dataset
def tokenize(item):
    text = make_prompt(item['description'], item['category'])
    tokens = tokenizer(text, truncation=True, max_length=MAX_LENGTH, padding='max_length')
    tokens['labels'] = tokens['input_ids'].copy()
    return tokens

tokenized_train = dataset['train'].map(tokenize, remove_columns=dataset['train'].column_names)
tokenized_val   = dataset['validation'].map(tokenize, remove_columns=dataset['validation'].column_names)

tokenized_train.set_format('torch')
tokenized_val.set_format('torch')

print(f"Tokenized train: {len(tokenized_train)} examples")

### Base model — before fine-tuning

Let's see how the untuned Llama performs. Expect ~25% — random guessing across 4 classes.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)
print(f"Base model loaded: {BASE_MODEL}")

In [ ]:
def predict(model, description: str) -> str:
    """Run inference and extract the ticket category."""
    prompt = make_prompt(description)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Extract the first matching category from the response
    for category in TICKET_CATEGORIES:
        if category.lower() in response.lower():
            return category
    return "Unknown"

# Quick test
test_case = "I was charged twice for my last invoice and need a refund."
print(f"Ticket: {test_case}")
print(f"Base model prediction: {predict(base_model, test_case)}")
print(f"Expected: Billing")

In [ ]:
from sklearn.metrics import accuracy_score

# Evaluate base model on 100 test tickets
test_sample = list(dataset['test'])[:100]
base_preds = [predict(base_model, item['description']) for item in test_sample]
base_labels = [item['category'] for item in test_sample]
base_accuracy = accuracy_score(base_labels, base_preds)

print(f"Base model accuracy (100 test tickets): {base_accuracy:.1%}")
print(f"Random baseline would be: 25%")

---

# Part 3 — Fine-Tune with QLoRA

**Requires T4 GPU.** Select Runtime → Change runtime type → T4 GPU before running this section.

We add LoRA adapter matrices to the attention layers of the 4-bit quantised model. Only 0.44% of parameters are trained.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,                   # rank — lower than Ed's 32 since our task is simpler
    lora_alpha=32,          # scaling factor = 2 × rank
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Reload model fresh (clean state before fine-tuning)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="ticket_model",
    num_train_epochs=3,              # 3 epochs — task is simple, converges fast
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch = 16
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    report_to="none",
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
)

print("Starting training...")
trainer.train()

In [ ]:
# Push fine-tuned adapters and tokenizer to HuggingFace Hub (private)
model.push_to_hub(MODEL_NAME, private=True)
tokenizer.push_to_hub(MODEL_NAME, private=True)
print(f"Fine-tuned model saved to: https://huggingface.co/{MODEL_NAME}")

---

# Part 4 — Evaluate

Full evaluation on the held-out test set: accuracy, F1-score per class, confusion matrix.

In [ ]:
from peft import PeftModel

# Load from Hub (use this cell if resuming a session)
# base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant_config, device_map="auto")
# model = PeftModel.from_pretrained(base_model, MODEL_NAME)

# Evaluate fine-tuned model on full test set
test_items = list(dataset['test'])
ft_preds  = [predict(model, item['description']) for item in test_items]
ft_labels = [item['category'] for item in test_items]

from sklearn.metrics import accuracy_score, classification_report
ft_accuracy = accuracy_score(ft_labels, ft_preds)
print(f"Fine-tuned model accuracy: {ft_accuracy:.1%}")
print()
print(classification_report(ft_labels, ft_preds, target_names=TICKET_CATEGORIES))

In [ ]:
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(ft_labels, ft_preds, labels=TICKET_CATEGORIES)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_pct, annot=True, fmt='.1f', cmap='Blues',
    xticklabels=TICKET_CATEGORIES, yticklabels=TICKET_CATEGORIES, ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (%) — Fine-tuned Llama Ticket Classifier')
plt.tight_layout()
plt.show()

# Key check: misclassification rate for Billing tickets
billing_idx = TICKET_CATEGORIES.index('Billing')
billing_misclass = 100 - cm_pct[billing_idx, billing_idx]
print(f"Misclassification rate for Billing tickets: {billing_misclass:.1f}%")
print("(Tickets classified as other than Billing — important for financial issues)")

---

# Part 5 — Model Comparison + Gradio UI

Compare three approaches:
1. Base Llama 3.2 3B (no training)
2. GPT-4.1-mini zero-shot (cloud, requires internet)
3. Fine-tuned Llama 3.2 3B with QLoRA (runs offline)

In [ ]:
# GPT-4.1-mini zero-shot on same test set
def predict_gpt(description: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": f"Classify this support ticket:\n{description}"}
        ],
        max_tokens=10,
        temperature=0
    )
    text = response.choices[0].message.content.strip()
    for category in TICKET_CATEGORIES:
        if category.lower() in text.lower():
            return category
    return "Unknown"

# Evaluate on 100 test tickets (to keep cost low)
eval_sample = test_items[:100]
eval_labels = [item['category'] for item in eval_sample]

gpt_preds  = [predict_gpt(item['description']) for item in eval_sample]
gpt_accuracy = accuracy_score(eval_labels, gpt_preds)
print(f"GPT-4.1-mini zero-shot accuracy: {gpt_accuracy:.1%}")

In [ ]:
# Evaluate base and fine-tuned on the same 100 tickets for fair comparison
base_preds_100 = [predict(base_model, item['description']) for item in eval_sample]
ft_preds_100   = [predict(model, item['description']) for item in eval_sample]

base_acc_100 = accuracy_score(eval_labels, base_preds_100)
ft_acc_100   = accuracy_score(eval_labels, ft_preds_100)

print(f"Base Llama accuracy:          {base_acc_100:.1%}")
print(f"GPT-4.1-mini zero-shot:       {gpt_accuracy:.1%}")
print(f"Fine-tuned Llama (QLoRA):     {ft_acc_100:.1%}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

models = ["Base Llama\n3.2 3B", "GPT-4.1-mini\n(zero-shot)", "Fine-tuned Llama\n3.2 3B (QLoRA)"]
accuracies = [base_acc_100 * 100, gpt_accuracy * 100, ft_acc_100 * 100]
colors = ["darkred", "slateblue", "steelblue"]
labels = ["Base (no training)", "Frontier (cloud API)", "Fine-tuned (offline)"]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(models, accuracies, color=colors, width=0.5)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.axhline(y=25, color='gray', linestyle='--', alpha=0.5, label='Random baseline (25%)')
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Support Ticket Classification — Model Comparison\n(Fine-tuned open source vs frontier, runs offline)', fontsize=13)
ax.set_ylim(0, 105)
ax.legend(fontsize=11)
legend_patches = [mpatches.Patch(color=c, label=l) for c, l in zip(colors, labels)]
ax.legend(handles=legend_patches + [plt.Line2D([0],[0], color='gray', linestyle='--', label='Random baseline')], fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nKey result: Fine-tuned Llama is {ft_acc_100 - gpt_accuracy:.1%} more accurate than GPT-4.1-mini")
print(f"And it runs OFFLINE — no customer data leaves the company.")

### Gradio UI

In [ ]:
import gradio as gr

CATEGORY_COLORS = {
    "Billing":         "#d32f2f",
    "Technical Issue": "#f57c00",
    "Account Management": "#1976d2",
    "General Inquiry":  "#388e3c",
    "Unknown":         "#9e9e9e",
}

CATEGORY_DESCRIPTIONS = {
    "Billing":         "BILLING — Issues related to payments, invoices, or charges.",
    "Technical Issue": "TECHNICAL ISSUE — Problems with product functionality or errors.",
    "Account Management": "ACCOUNT MANAGEMENT — Requests about account access, settings, or changes.",
    "General Inquiry":  "GENERAL INQUIRY — General questions or requests for information.",
    "Unknown":         "Unable to classify. Please review manually.",
}

def classify_ticket(description: str) -> str:
    if not description.strip():
        return "Please enter a support ticket description."
    category = predict(model, description.strip())
    color = CATEGORY_COLORS[category]
    desc  = CATEGORY_DESCRIPTIONS[category]
    return f'<div style="background:{color};color:white;padding:16px;border-radius:8px;font-size:18px;font-weight:bold">{desc}</div>'

examples = [
    ["I was charged twice for my last invoice and need a refund."],
    ["The app crashes every time I try to upload a file."],
    ["I forgot my password and can't log in."],
    ["What are your business hours?"],
    ["My subscription was cancelled without my request."],
    ["How do I update my billing address?"],
]

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Customer Support Ticket Classifier
    **Fine-tuned Llama 3.2 3B — runs offline, no customer data leaves the company**

    Enter a brief support ticket description as a customer would submit it.
    """)

    with gr.Row():
        inp = gr.Textbox(
            label="Support Ticket Description",
            placeholder="e.g. I can't access my account after the last update...",
            lines=3
        )
    btn = gr.Button("Classify", variant="primary")
    out = gr.HTML(label="Ticket Category")

    gr.Examples(examples=examples, inputs=inp)
    btn.click(classify_ticket, inputs=inp, outputs=out)

demo.launch()